# [Sensitivity Analysis] The Resolution Trade-Off (KAN-TabNet) | Step LR | Higgs Boson

**Optimized Reference Configuration:** `n_d=9, n_a=9`, `kan_grid_size=3`, `kan_spline_order=3`, `initial_lr=0.0025`

### Methodological Context: The Control Variables
To precisely isolate the effects of architectural parameter distribution, this sensitivity analysis uses the exact same continuous mathematical geometry (`k=3` cubic splines) and thermodynamic optimization environment (StepLR schedule, `initial_lr=0.0025`) as our optimized reference configuration.

By fixing the optimization environment, we strictly guarantee that any performance variations observed in this notebook are purely the result of the trade-off between the internal routing dimensions ($n_d, n_a$) and the B-spline grid resolution ($G$).

### The Hypothesis
In this study, we evaluate the "Configuration B" parameter distribution (`n_d=8, n_a=8`, `kan_grid_size=4`).

Because KAN layers are highly parameter-dense, researchers must constantly manage a strict parameter budget. This configuration tests a fundamental structural trade-off: sacrificing pipeline width (dropping from $9 \times 9$ to $8 \times 8$) in order to "buy" a higher mathematical resolution in the grid (increasing from $G=3$ to $G=4$), while maintaining a nearly identical total parameter footprint to our reference model.

We hypothesize that for continuous physics data containing inherent quantum noise, this is the wrong architectural trade-off. By narrowing the pipeline, we restrict the network's bandwidth to process the 28 physical features simultaneously. Concurrently, by increasing the grid resolution, we give the splines more mathematical flexibility to overfit the dataset's noise. This notebook serves to empirically prove that maximizing routing width while aggressively regularizing the grid is the mathematically superior approach for this dataset.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Notes:
# - momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 37 epochs approximates the paper's 20k iterations
#   (based on a batch size of 16384 and ~8.8M training samples).
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=37, gamma=0.9),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=8,
    n_a=8,
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025),
    use_kan=True,
    kan_grid_size=4,
    kan_spline_order=3,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.63548 | valid_accuracy: 0.68418 |  0:01:50s
epoch 25 | loss: 0.51269 | valid_accuracy: 0.74366 |  0:47:05s
epoch 50 | loss: 0.49371 | valid_accuracy: 0.75548 |  1:32:21s
epoch 75 | loss: 0.49073 | valid_accuracy: 0.75734 |  2:17:34s
epoch 100| loss: 0.48838 | valid_accuracy: 0.75876 |  3:02:48s
epoch 125| loss: 0.48354 | valid_accuracy: 0.76142 |  3:48:04s
epoch 150| loss: 0.481   | valid_accuracy: 0.76264 |  4:33:14s
epoch 175| loss: 0.47934 | valid_accuracy: 0.76373 |  5:18:27s
epoch 200| loss: 0.47848 | valid_accuracy: 0.76428 |  6:03:41s
epoch 225| loss: 0.47793 | valid_accuracy: 0.76471 |  6:48:52s
epoch 250| loss: 0.47788 | valid_accuracy: 0.76487 |  7:34:09s
epoch 275| loss: 0.47771 | valid_accuracy: 0.76457 |  8:19:25s
epoch 300| loss: 0.47689 | valid_accuracy: 0.76545 |  9:04:38s
epoch 325| loss: 0.47662 | valid_accuracy: 0.76543 |  9:49:52s
epoch 350| loss: 0.47621 | valid_accuracy: 0.7656  |  10:35:06s
epoch 375| loss: 0.4758  | valid_accuracy: 0.76516 |  

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 71,032


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.76626


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/06_kan_sensitivity_8x8_grid_4_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/06_kan_sensitivity_8x8_grid_4_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/06_kan_sensitivity_8x8_grid_4_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/06_kan_sensitivity_8x8_grid_4_71k.zip
